<a href="https://colab.research.google.com/github/kong230499-prog/ME-548/blob/main/06_stochastic_dynamic_programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stochastic dynamic programming
*(Adapted from Stanford AA 203)*

In this problem, we will examine discrete-time dynamic programming for stochastic systems—systems in which the outcome of a chosen action is not deterministic, but can take on multiple possible results, each occurring with a known probability. In such settings, we cannot directly optimize the value function, because even following a fixed sequence of actions can lead to different outcomes due to randomness. Instead, our goal is to optimize the [_expected value_](https://en.wikipedia.org/wiki/Expected_value) of the value function, averaging over all possible outcomes weighted by their probabilities. (If you could use a refresher on expected value, or have never studied it before, the linked Wikipedia article is a helpful starting point.)





Consider a machine that can be in one of two possible states at the start of each week: **running** or **broken**. The machine operates in weekly cycles. If it runs without failure throughout a week, it generates a gross profit of \$100. If it fails during the week, the gross profit for that week is \$0.

If the machine is **running** at the start of the week, you have the option to perform preventive maintenance. Performing maintenance reduces the probability of failure during the week to **0.4**, but comes at a cost of \$20. If you skip maintenance, the probability that the running machine fails during the week increases to **0.7**.

If the machine is **broken** at the start of the week, you may:
- **Repair** the machine for a cost of \$40. The repair restores it to the running state, but the probability it fails that week is still **0.4**.
- **Replace** the machine with a brand new one at a cost of \$150; a new machine is guaranteed to run successfully through its first week (probability of failure is 0 for the remainder of that week).

Additional notes:
- Performing maintenance on a broken machine does **not** repair it; the machine remains broken and the maintenance cost is wasted.
- Similarly, attempting to repair a machine that is already running has no effect (and you still pay the repair cost).

Your goal is to formulate and solve the optimal decision policy for maintaining, repairing, or replacing the machine each week to maximize the expected net profit over time.


In this notebook, you will set up all the pieces of the problem and perform dynamic programming to compute the optimal policy.

In [1]:
import numpy as np # no strong need to use jax in this notebook.
import matplotlib.pyplot as plt

## (a) Define the state space

The first step is to clearly define the possible states for our problem.
In Markov Decision Processes (MDPs), unlike traditional state-space models where states are often represented by continuous variables,
states can also take the form of discrete categories.
For this machine maintenance scenario, each state will represent a distinct condition or mode that the machine can be in at the start of a week, and we can represent that mode with a string.


In [2]:
# TODO: create a list of strings that represent all the possible states the machine can be in.

###### add your code here
state_space = ["running", "broken"]
###### end of add your code here



## (b) Define the action space

Similarly, define the action space of the machine maintenence problem.
*NOTE: actions are synonymous with controls. Instead of $u$ for controls, we use $a$ for actions.*

In [3]:
# TODO: create a list of strings that represent all the possible actions that can be taken.

###### add your code here
action_space = ["maintenance", "no_maintenance", "repair", "replace"]
###### end of add your code here



## (c) Define the transition probability matrix
In this step, we will specify how the system transitions from one state to another under each possible action.

Suppose we have a set of finite, discrete states $\{s_1, s_2, \ldots, s_N\}$ and a set of actions $\{a_1, a_2, \ldots, a_M\}$.
Let $T_{ijk}$ represent the probability of moving from state $s_i$ to state $s_j$ when action $a_k$ is taken. In other words,

$$
    T_{ijk} = \mathbb{P}(s_{\text{next}} = s_j \mid s_{\text{current}} = s_i,\, \text{action} = a_k)
$$

For convenience, you should represent the transition probability matrices using a Python dictionary.
Each dictionary key will be one of the actions, and the corresponding value will be the $N \times N$ transition matrix for that action.
Each row of a matrix should sum to $1$, representing the probabilities of transitioning from each current state to any next state, given the chosen action.


In [4]:
# TODO: create a dictionary of transition matrices for each action
# NOTE: The ordering of the states in the transition matrices should match the ordering of the states in the state_space list.
# e.g., row i in the matrix should correspond to state i in the state_space list.
###### add your code here
transition_matrices = {}

# Maintenance (only meaningful if running)
transition_matrices["maintenance"] = np.array([
    [0.6, 0.4],  # running → running / broken
    [0.0, 1.0],  # broken stays broken
])

# No maintenance
transition_matrices["no_maintenance"] = np.array([
    [0.3, 0.7],
    [0.0, 1.0],
])

# Repair
transition_matrices["repair"] = np.array([
    [0.3, 0.7],  # repair on running = waste → behaves like no maintenance
    [0.6, 0.4],  # repaired becomes running, with failure prob 0.4
])

# Replace
transition_matrices["replace"] = np.array([
    [1.0, 0.0],  # guaranteed success week
    [1.0, 0.0],
])
###### end of add your code here



In [5]:
# The functions below are provided for you, you do not need to change them
# Take a look at them to understand how the transition matrices are used, as well as the state and action spaces.


def get_transition_probability(s_current: str, s_next: str, action: str):
    """Returns the transition probability of moving from s_current to s_next when taking action."""
    action_matrix = transition_matrices[action]
    s_current_index = state_space.index(s_current)
    s_next_index = state_space.index(s_next)
    return action_matrix[s_current_index, s_next_index]

def get_transition_probability_all(s_current: str, action: str):
    """Returns the transition probabilities of moving from s_current to all next states when taking action."""
    action_matrix = transition_matrices[action]
    s_current_index = state_space.index(s_current)
    return action_matrix[s_current_index, :]

def sample_next_state(s_current: str, action: str, k: int = 1):
    """Given a current state and an action, sample the next state based on transition probabilities.
    Arguments:
        s_current: current state
        action: action taken
        k: number of samples to draw
    Returns:
        list of sampled next states
    """
    probabilities = get_transition_probability_all(s_current, action)
    return [state_space[i] for i in np.random.choice(range(len(state_space)), p=probabilities, size=k).tolist()]

## (d) Write out the state-action-next-state reward function

Define the reward function $r(s_t, a_t, s_{t+1})$ for all possible combinations of states, actions, and resulting next states.

Specifically, based on the states, actions, and transition definitions provided above,
enumerate the numerical reward values assigned for each $(s_t, a_t, s_{t+1})$ triplet.
Your answer should make clear how the reward depends on the current state, action taken, and the resulting next state.


In [6]:
# TODO: create a dictionary of reward matrices for each action
# NOTE: The ordering of the states in the reward matrices should match the ordering of the states in the state_space list.
# e.g., row i in the matrix should correspond to state i in the state_space list.
###### add your code here
reward_matrices = {}

# Maintenance
reward_matrices["maintenance"] = np.array([
    [100 - 20, 0 - 20],
    [0 - 20, 0 - 20],
])

# No maintenance
reward_matrices["no_maintenance"] = np.array([
    [100, 0],
    [0, 0],
])

# Repair
reward_matrices["repair"] = np.array([
    [100 - 40, 0 - 40],
    [100 - 40, 0 - 40],
])

# Replace
reward_matrices["replace"] = np.array([
    [100 - 150, 100 - 150],
    [100 - 150, 100 - 150],
])
###### end of add your code here



In [7]:
# The function below are provided for you, you do not need to change them
# Take a look at them to understand how the reward matrices are used, as well as the state and action spaces.
def get_reward(s_current, s_next, action):
    """Returns the reward of moving from s_current to s_next when taking action."""
    reward_matrix = reward_matrices[action]
    s_current_index = state_space.index(s_current)
    s_next_index = state_space.index(s_next)
    return reward_matrix[s_current_index, s_next_index]

## (e) Run dynamic programming to find the optimal policy!

In this part, you will use dynamic programming to determine the optimal sequence of repair, replacement, and maintenance actions that maximize the total expected profit over a four-week planning horizon.

Assume the machine is new at the beginning of the first week (Week 0). Clearly indicate how you formulate and solve for the optimal policy at each time step.




### (i) Define the Terminal Value Function (V_terminal)
Before running dynamic programming, we need to initialize our value function table, which stores the expected value of each state at every time step.
Our planning horizon covers five time steps, corresponding to the *start* of each week:

Week 0 (t=0), Week 1 (t=1), Week 2 (t=2), Week 3 (t=3), Week 4 (t=4), Week 5 (t=5)

Remember: Rewards are only earned for successfully completing a week. There is no reward simply for being in a specific state at the start of a week.
Based on this, specify what the value function should be for each possible state at the final time step (the terminal time, t=5). This will serve as the boundary condition for your dynamic programming computation.


In [8]:
# TODO: Define the terminal value function
###### add your code here
V_terminal = np.array([0.0, 0.0])
###### end of add your code here


# Below is a function that will be used in the dynamic programming algorithm.
# You do not need to change it.

def get_value(state, V_table):
    """Given a state and a value function table, return the value of the state."""
    state_index = state_space.index(state)
    return V_table[state_index]


### (ii) Implement the Bellman Update

At this stage, you have all the components needed to carry out dynamic programming. Your next step is to perform the Bellman update at each time step as you work backward through time.

Complete the `bellman_update` function below. This function should update the value function for each state by considering all possible actions, transitions, and expected rewards, choosing the optimal action at each state.


In [9]:
def bellman_update(value: np.ndarray, discount_factor: float = 1.0) -> tuple[np.ndarray, dict]:
    """Perform a Bellman update to compute the next value function table.
    Args:
        value: current value function table
    Returns:
        value_back: updated value function table, one step backward in time
        policy: optimal action for each state

    value_back[s_current] = max over actions of [ sum over s_next of ( reward(s_current, s_next, action) + discount_factor * value[s_next] ) ]
    """

    # initialize the updated value function and policy
    value_back = np.zeros_like(value)
    policy = {}

    # TODO: implement the Bellman update


    ####### add your code here

    for s in state_space:
        best_value = -np.inf
        best_action = None

        for a in action_space:
            probs = get_transition_probability_all(s, a)

            expected_value = 0
            for i, s_next in enumerate(state_space):
                r = get_reward(s, s_next, a)
                expected_value += probs[i] * (r + discount_factor * value[i])

            if expected_value > best_value:
                best_value = expected_value
                best_action = a

        value_back[state_space.index(s)] = best_value
        policy[s] = best_action

    ####### end of add your code here



    return value_back, policy


In [10]:
V_next, policy = bellman_update(V_terminal)  # test the function
V_next, policy

(array([40., 20.]), {'running': 'maintenance', 'broken': 'repair'})

### (iii) Now run the Bellman update over the time horizon!

Now, you can perform the Bellman update over multiple timesteps. At each time step, you should collect the value function and the policy.

In [11]:
# TODO: Perform the Bellman update over the time horizon.

value_function = {} # a dictionary of value functions, indexed by time step
policy_list = {} # a list of policies, indexed by time step

###### add your code here

T = 5  # horizon

value_function[T] = V_terminal

for t in reversed(range(T)):
    V_next, policy = bellman_update(value_function[t+1])
    value_function[t] = V_next
    policy_list[t] = policy

###### end of add your code here



### (iv) Verify your value function answer
Now that you have computed the optimal value function, you can verify that value matches the expected value from many random simulations when executing the optimal policy.

Run a number of trials whilst following the optimal policy and see what the average reward received.


In [12]:
np.random.seed(1)  # for reproducibility

# TODO: perform several trials following the optimal policy and compute the average reward
###### add your code here

n_trials = 1000
total_rewards = []

for _ in range(n_trials):
    state = "running"  # initial condition
    total_reward = 0

    for t in range(T):
        action = policy_list[t][state]

        next_state = sample_next_state(state, action)[0]

        reward = get_reward(state, next_state, action)
        total_reward += reward

        state = next_state

    total_rewards.append(total_reward)

print("Average reward:", np.mean(total_rewards))
print("DP value estimate:", value_function[0][state_space.index("running")])

###### end of add your code here



Average reward: 168.72
DP value estimate: 168.0
